# S³ score distributions by BERTopic topic × S³ axis

This plotting-only notebook reads the already-computed k=10, seed=42 S³ passage scores. It does not refit S³, recompute embeddings, or change BERTopic assignments. Each box is a distribution of passage scores for one original BERTopic topic and one S³ axis. Topic -1 (BERTopic outliers) is included. Scores are not probabilities.

The matrix is split into pages of 20 topic rows so all topics remain visible without creating one extremely tall figure.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
scores_path = project_root / 'data/topic-analysis/results/jev-semantic-all/s3-topic-axis-distributions/passage-scores-k10-seed42.parquet'
if not scores_path.exists():
    raise FileNotFoundError(f'Run the k=10 seed=42 S³ notebook first; expected saved scores at {scores_path}')
scored_passages = pd.read_parquet(scores_path)
score_columns = [column for column in scored_passages.columns if column.startswith('s3_component_')]
score_columns = sorted(score_columns, key=lambda column: int(column.rsplit('_', 1)[1]))
if not score_columns:
    raise ValueError(f'No S³ component score columns found in {scores_path}')
topic_order = sorted(scored_passages.topic_id.dropna().unique().tolist())
score_frame = scored_passages[score_columns]
score_std = score_frame.std(ddof=0).replace(0, 1)
score_z = (score_frame - score_frame.mean()) / score_std
print(f'{len(scored_passages):,} passages | {len(topic_order)} BERTopic groups | {len(score_columns)} S³ axes | outliers: {(scored_passages.topic_id == -1).sum():,}')
print('Saved S³ scores loaded; no embeddings or models were recomputed.')

4,132 passages | 237 BERTopic groups | 10 S³ axes | outliers: 827
Saved S³ scores loaded; no embeddings or models were recomputed.


In [2]:
topics_per_page = 20
for page_start in range(0, len(topic_order), topics_per_page):
    page_topics = topic_order[page_start:page_start + topics_per_page]
    fig = make_subplots(
        rows=len(page_topics), cols=len(score_columns),
        shared_xaxes=False, shared_yaxes=False,
        horizontal_spacing=0.025, vertical_spacing=min(0.02, 0.3 / max(1, len(page_topics))),
        subplot_titles=[f'S³ axis {axis_id}' for _topic in page_topics for axis_id in range(len(score_columns))],
    )
    for row, topic_id in enumerate(page_topics, start=1):
        topic_mask = scored_passages.topic_id.to_numpy() == topic_id
        for col, score_column in enumerate(score_columns, start=1):
            fig.add_trace(
                go.Box(
                    x=score_z.loc[topic_mask, score_column],
                    y=[str(topic_id)] * int(topic_mask.sum()),
                    orientation='h', boxpoints=False, showlegend=False,
                    hovertemplate=f'Topic {topic_id}<br>standardized S³ score=%{{x:.2f}}<extra></extra>',
                ), row=row, col=col,
            )
        fig.update_yaxes(title_text=f'Topic {topic_id}', showticklabels=False, row=row, col=1)
    fig.update_xaxes(title_text='Standardized S³ score (not a probability)')
    fig.update_layout(
        title=f'S³ score distributions by original BERTopic topic × axis (topics {page_topics[0]}–{page_topics[-1]})',
        height=max(900, 105 * len(page_topics)), showlegend=False,
        margin={'l': 100, 'r': 25, 't': 90, 'b': 65},
    )
    fig.show()